In [56]:
import asyncio
import openai
import sendgrid
import os

from openai import AsyncOpenAI
from sendgrid.helpers.mail import Mail, Email, To, Content
from agents import Agent, Runner, OpenAIChatCompletionsModel, GuardrailFunctionOutput, InputGuardrailTripwireTriggered
from agents import function_tool, trace, handoff, input_guardrail
from dotenv import load_dotenv
from pydantic import BaseModel


GPT_MODEL = 'gpt-5-nano'

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"


params = {
    'from_email': 'yuko.sgp@gmail.com',
    'to_email': 'kageyama.sgp@gmail.com'
}


load_dotenv()

True

In [12]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [13]:
writer_instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing \
for audits, powered by AI. \
You write professional, serious cold emails. \
Use only the information explicitly provided in the input \
Do not invent, assume, or infer any missing details \
If information is missing, omit it \
Output must be a complete email that do not require further editing."

writer_instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response. \
Use only the information explicitly provided in the input \
Do not invent, assume, or infer any missing details \
If information is missing, omit it \
Output must be a complete email that do not require further editing."

writer_instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails.\
Use only the information explicitly provided in the input \
Do not invent, assume, or infer any missing details \
If information is missing, omit it \
Output must be a complete email that do not require further editing."


deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model='gemini-2.5-flash-lite', openai_client=gemini_client)
groq_model = OpenAIChatCompletionsModel(model='llama-3.3-70b-versatile', openai_client=groq_client)


sales_agent1 = Agent(
    name='Deepseek Sales Agent',
    instructions=writer_instructions1,
    model=deepseek_model,
)

sales_agent2 = Agent(
    name="Gemini Sales Agent",
    instructions=writer_instructions2,
    model=gemini_model,
)

sales_agent3 = Agent(
    name='Groq Sales Agent',
    instructions=writer_instructions3,
    model=groq_model,
)

In [14]:
description = 'Write a cold sales email'

tool_writer1 = sales_agent1.as_tool(tool_name='sales_agent1', 
                               tool_description=description)
tool_writer2 = sales_agent2.as_tool(tool_name='sales_agent2', 
                               tool_description=description)
tool_writer3 = sales_agent3.as_tool(tool_name='sales_agent3', 
                               tool_description=description)

In [19]:
@function_tool
def send_email(subject: str, body: str):
    """Send out an email with the given subject, and body to all sales prospects."""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email(params['from_email'])
    to_email = To(params['to_email'])
    content = Content("text/html", body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": 'success'}

In [21]:
html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

html_converter = Agent(
    name='HTML email body converter', 
    instructions=html_instructions,
    model=GPT_MODEL,
)

html_tool = html_converter.as_tool(tool_name='html_converter',
            tool_description="Converts text email body into an HTML email body")
tools = [html_tool, send_email]

emailer_instructions = "You are an email formatter and sender. Your goal is to convert email body to HTML body and send the email.\
Follow these steps carefully: \
1. You use the html_converter tool to convert the email body to HTML. \
2. You use the send_email tool to send the email with the subject and HTML body. \
3. Only return completed steps and status, do not include the email sent."

emailer_agent = Agent(
    name = 'Email Manager',
    instructions=emailer_instructions,
    tools=tools,
    model=GPT_MODEL,
    handoff_description="Convert an email to HTML and send it",
)

In [22]:
emailer_tool = emailer_agent.as_tool(tool_name='emailer_agent', tool_description='Convert an email to HTML and send it')
emailer_handoff = handoff(
    agent=emailer_agent,
    tool_name_override='transfer_to_email_manager',
)

In [33]:
manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to send the single best cold sales email.

Follow these steps carefully:
1. Call all three sales_agent tools simultaneously to generate three email drafts in parallel.
2. Choose the single best email based on: subject line strength, personalisation, 
   concise value proposition, clear call to action, and tone.
3. Invoke the handoff tool with this payload:
   {
     "subject"" "<subject of the chosen draft>"
     "email_body": "<full text of the chosen draft>"
   }
"""

sales_manager = Agent(
    name='Sales Manager',
    instructions=manager_instructions,
    model=GPT_MODEL,
    tools=[tool_writer1, tool_writer2, tool_writer3],
    handoffs=[emailer_handoff],
)
    
query = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("MultiModel Automated SDR"):
    response = await Runner.run(sales_manager, query)

print(response.final_output)

Completed steps and status:

- Generated 3 draft emails (sales_agent1, sales_agent2, sales_agent3): completed
- Selected preferred draft (sales_agent2): completed
- Transferred selected draft to Email Manager: completed
- Converted email body to HTML (html_converter): completed
- Sent email to all sales prospects (send_email): success


In [49]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

In [58]:
name_guardrail = Agent(
    name='Name Guardarail check',
    instructions="Check if the user is including someone's personal name in what they want to do.",
    output_type=NameCheckOutput,
    model=GPT_MODEL,
    
)

@input_guardrail
async def guardrails_against_name(ctx, agent, message):
    result = await Runner.run(name_guardrail, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"guardrail_payload": result.final_output},
                                  tripwire_triggered=is_name_in_message)


careful_sales_agent = Agent(
    name='Sales Manager',
    instructions=manager_instructions,
    model=GPT_MODEL,
    tools=[tool_writer1, tool_writer2, tool_writer3],
    handoffs=[emailer_handoff],
    input_guardrails=[guardrails_against_name]
)

message = 'Send out a cold sales email addressed to Dear CEO'

try:
    with trace('Protected Automated SDR'):
        result = await Runner.run(careful_sales_agent, message)
except InputGuardrailTripwireTriggered as e:
    print(f"Error: {e}")
    

In [59]:
print(result.final_output)

Step 1: Converted email body to HTML — Completed
Step 2: Sent email with subject "Scale efficiently: 15-minute plan to boost team productivity" — Completed
